In [ ]:
# Import all required libraries
import os
import shutil
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
# Load the CSV file containing file names and labels
# Replace 'path to csv' with the actual path to your CSV file
df = pd.read_csv("path to csv")

In [ ]:
# ===== STEP 1: Identify unique classes and create directory structure =====

# Get unique classes from your label column
# Replace 'label_column' with your actual label column name
unique_classes = df['label_column'].unique()

print(f"Unique classes found: {unique_classes}")
print(f"Total classes: {len(unique_classes)}\n")

# Create a base directory for class folders
base_dir = "class_wise_data"  # You can change this path
os.makedirs(base_dir, exist_ok=True)

# Create folders for each class with train/test/val subfolders
# This structure allows organized data splits per class
splits = ['train', 'test', 'val']
for class_name in unique_classes:
    class_dir = os.path.join(base_dir, str(class_name))
    for split in splits:
        split_dir = os.path.join(class_dir, split)
        os.makedirs(split_dir, exist_ok=True)
        print(f"Created directory: {split_dir}")

print(f"\nAll directories created successfully in '{base_dir}'")

In [ ]:
# ===== STEP 2: Split data and organize files by class and split =====

# Configuration parameters - UPDATE THESE with your actual column names
file_column = 'file_name'  # Column containing file names/paths
label_column = 'label_column'  # Column containing class labels
source_dir = 'source_data'  # Directory where original files are located

# Define train/test/val split ratios
# These add up to 1.0 (70% train, 15% test, 15% validation)
train_ratio = 0.7
test_ratio = 0.15
val_ratio = 0.15

# Process each class separately to maintain class distribution
for class_name in unique_classes:
    # Get all files for the current class
    class_data = df[df[label_column] == class_name].copy()
    
    # First split: Separate train (70%) from temp (30% for test+val)
    train_files, temp_files = train_test_split(
        class_data, 
        test_size=(test_ratio + val_ratio),
        random_state=42  # Fixed seed for reproducibility
    )
    
    # Second split: Divide temp into test (50%) and val (50%)
    # This ensures test and val each get 15% of the original data
    test_files, val_files = train_test_split(
        temp_files,
        test_size=0.5,
        random_state=42
    )
    
    # Print split summary for this class
    print(f"\nClass '{class_name}':")
    print(f"  Train: {len(train_files)} files")
    print(f"  Test:  {len(test_files)} files")
    print(f"  Val:   {len(val_files)} files")
    
    # Group files by their split (train/test/val)
    splits_data = [
        ('train', train_files),
        ('test', test_files),
        ('val', val_files)
    ]
    
    # Copy files to their designated directories
    for split_name, split_df in splits_data:
        for idx, row in split_df.iterrows():
            file_name = row[file_column]
            
            # Construct source and destination paths
            source_path = os.path.join(source_dir, file_name)
            destination_dir = os.path.join(base_dir, str(class_name), split_name)
            destination_path = os.path.join(destination_dir, Path(file_name).name)
            
            # Check if source file exists before copying
            if os.path.exists(source_path):
                try:
                    # Copy file preserving metadata (timestamps, permissions)
                    shutil.copy2(source_path, destination_path)
                except Exception as e:
                    print(f"  ✗ Error copying {file_name}: {e}")
            else:
                print(f"  ✗ Source file not found: {source_path}")

print("\n✓ Data organization complete!")